In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score

# 1. Sample Dataset Generate Karein
np.random.seed(42)
n = 500
data = {
    'Order ID': [f'ORD{1000+i}' for i in range(n)],
    'Customer ID': [f'CUST{np.random.randint(100, 150)}' for _ in range(n)],
    'Order Date': pd.date_range(end=pd.Timestamp.today(), periods=n, freq='D'),
    'Revenue': np.random.uniform(50, 1000, n),
    'Profit': np.random.uniform(10, 200, n)
}
df = pd.DataFrame(data)

# 2. Data Cleaning
df = df.drop_duplicates(subset=['Order ID']).dropna()

# 3. RFM Metrics Calculation
max_date = df['Order Date'].max()
rfm = df.groupby('Customer ID').agg({
    'Order Date': lambda x: (max_date - x.max()).days,
    'Order ID': 'nunique',
    'Revenue': 'sum'
}).reset_index()

rfm.columns = ['Customer ID', 'Recency', 'Frequency', 'Monetary']
rfm['Churn'] = np.where(rfm['Recency'] >= 180, 1, 0)

# 4. Machine Learning Model Training
X = rfm[['Recency', 'Frequency', 'Monetary']]
y = rfm['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression()
model.fit(X_train, y_train)

# 5. Risk Category Stratification
rfm['Churn_Probability'] = model.predict_proba(X)[:, 1]

def get_risk_bucket(prob):
    if prob >= 0.70:
        return 'High Risk'
    elif prob >= 0.40:
        return 'Medium Risk'
    else:
        return 'Low Risk'

rfm['Risk_Category'] = rfm['Churn_Probability'].apply(get_risk_bucket)

print("--- Data Analytics & Model Execution Successful ---")
print(rfm.head(10))

--- Data Analytics & Model Execution Successful ---
  Customer ID  Recency  Frequency     Monetary  Churn  Churn_Probability  \
0     CUST100       24         12  6220.971577      0       3.096615e-12   
1     CUST101       75         14  7618.915401      0       1.181821e-06   
2     CUST102       18         12  5523.314258      0       2.475914e-13   
3     CUST103       56          8  4695.512912      0       1.016390e-10   
4     CUST104        4          9  3407.613356      0       2.692112e-16   
5     CUST105       55          9  4217.898706      0       3.356216e-11   
6     CUST106       89          9  3683.761283      0       1.084927e-08   
7     CUST107       61         12  7499.586461      0       5.734856e-08   
8     CUST108       35         10  5456.947509      0       6.515593e-12   
9     CUST109      425          1   650.245352      1       1.000000e+00   

  Risk_Category  
0      Low Risk  
1      Low Risk  
2      Low Risk  
3      Low Risk  
4      Low Risk  
5  